# Drive a virtual robot with Aiko Turtle

*This notebook was written for Google Colab but has not been run there yet. If something breaks, please report it as an issue on the repo: https://github.com/anjsimmo/aiko_turtle*

In about 10 minutes you will:

1. Drive a robot over a plain web API (no installs).
2. Run your own robot over Aiko Services and talk to it in chat.
3. Ask an AI agent to draw a shape for you.

Open the live world in another tab so you can watch your robot move: **https://turtle.aiko.hashtag.ai/world**

The world is shared with everyone at the meetup, so pick an **unusual robot name** (not "turtle"!).

Run the cells from top to bottom.

## Your settings

Change `ME` to your own name (no `@`, it is added for you) and `NAME` to a robot name nobody else will pick.

In [ ]:
ME = "your_name"            # your name, no @
NAME = "my_unusual_turtle"  # your robot's name, make it unique

BASE = "https://turtle.aiko.hashtag.ai"
TOKEN = ""                  # only needed if the instance asks for a token

assert ME != "your_name", "Please edit ME above first"
print(f"Hi @{ME}! Your robot will be called {NAME}.")
print(f"Watch it at {BASE}/world")

## Part 1: drive over REST (no install)

REST is just web requests. The helper below uses only Python's standard library. Every drive call returns the robot's pose: `x`, `y` and `heading`.

In [ ]:
import json
import urllib.request

def call(method, path, body=None):
    data = json.dumps(body).encode() if body is not None else None
    headers = {"Content-Type": "application/json"}
    if TOKEN:
        headers["Authorization"] = f"Bearer {TOKEN}"
    request = urllib.request.Request(BASE + path, data=data, method=method, headers=headers)
    with urllib.request.urlopen(request) as response:
        return json.loads(response.read())

print("Robots in the world:", [r["name"] for r in call("GET", "/api/robots")])
print("Created:", call("POST", "/api/robots", {"name": NAME, "kind": "turtle", "owner": "@" + ME}))

Now draw a square: move forward 100, turn 90 degrees counter-clockwise, four times. Watch the world page!

In [ ]:
USER = "@" + ME
for _ in range(4):
    call("POST", f"/api/robots/{NAME}/move", {"distance": 100, "username": USER})
    pose = call("POST", f"/api/robots/{NAME}/turn", {"angle": 90, "username": USER})
    print(f"  at x={pose['x']:g} y={pose['y']:g} heading={pose['heading']:g}")

Say something in the shared chat, and look at what your robot has done so far.

In [ ]:
call("POST", "/api/chat", {"username": USER, "message": f"{NAME} drew a square over REST", "channel": "robots"})
print("History:", [e["command"] for e in call("GET", f"/api/robots/{NAME}/history")])

## Part 2: the Aiko way

Under the hood the robots talk over [Aiko Services](https://github.com/geekscape/aiko_services) using MQTT. Here your notebook will run its **own** robot process and chat with it.

First, install the packages and get the project (this can take a minute or two).

> If Colab complains or restarts the session, an older fix was to remove Pillow and psutil from aiko_services' requirements before installing:
> `!git clone https://github.com/geekscape/aiko_services` then
> `!sed -i '/[Pp]illow/d; /psutil/d' aiko_services/pyproject.toml` then `!pip install ./aiko_services git+https://github.com/geekscape/aiko_chat`

In [ ]:
!pip install -q git+https://github.com/geekscape/aiko_services git+https://github.com/geekscape/aiko_chat python-dotenv sqlalchemy click
!git clone -q https://github.com/anjsimmo/aiko_turtle || echo "already cloned"

Now start your robot in the background. It connects to the shared Aiko services at `aiko.hashtag.ai`.

**While this process runs, your notebook is your robot's voice in the chat.** When you stop it (the Stop cell near the end), the web server takes over again.

In [ ]:
import os, subprocess, sys, threading, time

ENV = dict(os.environ, AIKO_MQTT_HOST="aiko.hashtag.ai", AIKO_MQTT_PORT="1883",
           AIKO_NAMESPACE="aiko", USER=ME)
os.environ.update(AIKO_MQTT_HOST="aiko.hashtag.ai", AIKO_MQTT_PORT="1883",
                  AIKO_NAMESPACE="aiko", USER=ME)

robot = subprocess.Popen(
    [sys.executable, "robot.py", "run", "--name", NAME, "--kind", "turtle"],
    cwd="aiko_turtle", env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

robot_output = []
def collect():
    for line in robot.stdout:
        robot_output.append(line.rstrip())
threading.Thread(target=collect, daemon=True).start()

time.sleep(5)
print("\n".join(l for l in robot_output if l.startswith("ROBOT")) or "(no output yet, wait a few seconds and run the next cell)")

Send chat commands to your robot. In the `robots` channel, a message that **starts with** `@@NAME` is a command for that robot. (Humans are `@name`, bots are `@@name`.) Commands: `move 100`, `turn 90`, `look`, `reset`, `telemetry`.

In [ ]:
def say(message, wait=3):
    """Send a chat message to #robots, then print what the robot printed in reply."""
    before = len(robot_output)
    subprocess.run(["aiko_chat", "send", "robots", message], env=ENV)
    time.sleep(wait)
    for line in robot_output[before:]:
        if line.startswith(f"ROBOT {NAME}:"):
            print(line)

say(f"@@{NAME} move 100")
say(f"@@{NAME} turn 90")
say(f"@@{NAME} look")

## Part 3: ask the AI agent

The LLM agent `@@pilot` is also listening in the chat. Ask it to drive your robot, then watch the world page.

In [ ]:
say(f"@@pilot draw a triangle with {NAME}", wait=20)
print(f"Watch it at {BASE}/world")

## Stop your robot
This ends your robot process, and the web server takes over the robot's voice again.

In [ ]:
robot.terminate()
try:
    robot.wait(timeout=10)
except subprocess.TimeoutExpired:
    robot.kill()
print("Robot stopped.")

## Where next?

- The repo README: https://github.com/anjsimmo/aiko_turtle
- `examples/aiko_drive.py`: drive robots over Aiko from your own script.
- `examples/agent_bot.py`: build your own agent bot.
- Connect a real robot at the **AI+ML+Robots** meetup: https://www.meetup.com/ai-ml-robots/ and https://ai-ml-robots.github.io/

Thanks for playing!